# 12 — Silver Events

**Objetivo:** canonicalizar la fuente `SECOP II - Adiciones` como una capa de eventos reutilizable y temporalmente tipada.

Silver preserva **todos** los valores de `Tipo`. No decide todavía si el label de Gold significa “cualquier evento de la fuente” o un subconjunto semántico específico.

## Estrategia out-of-core

`Adiciones` tiene decenas de millones de filas. Se usa staging Parquet + fingerprint estrecho + verificación de igualdad raw para deduplicar exactamente sin un `GROUP BY ALL` ancho sobre todo el CSV.

In [1]:
import tomllib
import pandas as pd
from argos.common.paths import ProjectPaths
from argos.config.registry import load_registry, discover_dataset_files
from argos.config.silver import load_silver_spec
from argos.silver.events import BUILD_IMPLEMENTATION

paths = ProjectPaths.discover(); paths.ensure_runtime_dirs()
with (paths.configs / "project.toml").open("rb") as handle:
    PROJECT_CONFIG = tomllib.load(handle)
DATASETS, _ = load_registry(paths.configs / "datasets.toml")
FILES = discover_dataset_files(paths.bronze, DATASETS)
SILVER_SPEC = load_silver_spec(paths.configs / "silver.toml")

check11 = pd.read_parquet(paths.reports / "silver" / "11_validation.parquet")
blocking11 = check11.loc[check11["blocking"]]
assert not blocking11.empty and blocking11["passed"].all(), "Notebook 11 no está cerrado."

{
    "additions": FILES["additions"].name,
    "implementation": BUILD_IMPLEMENTATION,
    "schema_version": SILVER_SPEC.schema_version,
    "duckdb_memory_limit": SILVER_SPEC.runtime.duckdb_memory_limit,
}

{'additions': 'SECOP_II_-_Adiciones_20260921.csv',
 'implementation': 'm2-12-addition-events-out-of-core-v2',
 'schema_version': 'm2-silver-v1',
 'duckdb_memory_limit': '8GB'}

## Construcción

- `addition_event_versions.parquet`: payloads Bronze exactos después de colapsar solo duplicados idénticos.
- `addition_events.parquet`: grain canónico del evento `(contract_id, event_at, event_type)` con diagnósticos de variantes y vínculo al contrato.

In [2]:
from argos.silver.events import build_events_silver, validate_existing_events_silver

_expected = [
    paths.silver / "addition_event_versions.parquet",
    paths.silver / "addition_events.parquet",
]
if all(p.exists() for p in _expected):
    print("[12] complete event artifact set detected; validating/finalizing without rebuild...")
    result = validate_existing_events_silver(
        reports_root=paths.reports,
        silver_dir=paths.silver,
        manifest_dir=paths.root / "artifacts" / "manifests",
        silver_spec=SILVER_SPEC,
        additions_path=FILES["additions"],
        temp_directory=paths.root / ".cache" / "duckdb",
    )
else:
    result = build_events_silver(
        FILES["additions"],
        additions_spec=DATASETS["additions"],
        reports_root=paths.reports,
        silver_dir=paths.silver,
        manifest_dir=paths.root / "artifacts" / "manifests",
        silver_spec=SILVER_SPEC,
        date_order=PROJECT_CONFIG["temporal"]["date_order"],
        sample_size=PROJECT_CONFIG["profiling"]["csv_sample_size"],
        temp_directory=paths.root / ".cache" / "duckdb",
        overwrite=True,
    )
{name: str(path.relative_to(paths.root)) for name, path in result.artifacts.items()}

[12] additions: staging raw payloads (streaming CSV -> Parquet)...
[12] additions: finding duplicate fingerprints (narrow aggregate)...
[12] additions: resolving candidate duplicates by exact raw equality...
[12] additions: materializing exact event versions...
[12] additions: materializing canonical event grain...


{'addition_event_versions': 'data/silver/addition_event_versions.parquet',
 'addition_events': 'data/silver/addition_events.parquet'}

## Validación

In [3]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,addition_versions_represent_source,events,True,True,"{""m1_expected"": 28378604, ""represented_source_...",exact-deduplicated event versions account for ...
1,addition_event_time_parseability,temporal,True,True,"{""missing_source_rows"": 0, ""parse_success_nonm...",addition event time parse success >= 99.00%
2,addition_event_grain,events,True,True,"{""distinct_event_ids"": 8636666, ""events"": 8636...",one row per canonical event id with complete s...
3,addition_contract_coverage,relationships,True,True,"{""coverage"": 0.9999182554934971, ""matched"": 86...",canonical addition→contract coverage >= 99.00%
4,addition_event_scope_preserved,events,True,True,"{""distinct_nonnull_event_types"": 10}",Silver preserves all event types; target seman...


In [4]:
from argos.common.duckdb import connect
from argos.common.sql import sql_string

con = connect(temp_directory=paths.root / ".cache" / "duckdb", threads=SILVER_SPEC.runtime.duckdb_threads, memory_limit=SILVER_SPEC.runtime.duckdb_memory_limit)
try:
    event_types = con.execute(f"""
        SELECT event_type, COUNT(*) AS canonical_events, SUM(source_row_count) AS source_rows
        FROM read_parquet({sql_string(result.artifacts['addition_events'])})
        GROUP BY event_type ORDER BY source_rows DESC
    """).fetchdf()
finally:
    con.close()
event_types

,event_type,canonical_events,source_rows
0,MODIFICACION GENERAL,3616766,13490619.0
1,CONCLUSION,1968834,7022896.0
2,No definido,2470594,6024441.0
3,ADICION EN EL VALOR,240204,561997.0
4,CESION,95980,477761.0
5,REACTIVACIoN,127463,429798.0
6,SUSPENSIoN,116340,370555.0
7,EXTENSION,352,391.0
8,REDUCCION EN EL VALOR,132,145.0
9,EXPIRACION,1,1.0


**Interpretación:** esta tabla describe la fuente, no redefine el target. El freeze semántico de `y_addition_180d` se hará en Gold y quedará documentado en el target spec.

In [5]:
blocking = result.validation.loc[result.validation["blocking"]]
failed = blocking.loc[~blocking["passed"], "check_id"].tolist()
print("12 ✅ PASS" if result.ready_for_next_notebook else "12 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
if failed:
    print("Failed:", ", ".join(failed))
assert result.ready_for_next_notebook, "Notebook 12 remains OPEN; resolve blocking checks before 13."

12 ✅ PASS
Blocking checks: 5/5
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/silver_events.json


**Siguiente:** `13_entity_resolution.ipynb`.